In [1]:
import os
import sys
import glob

# 1. Add base PySpark path
sys.path.insert(0, '/opt/spark/python')

# 2. Dynamically locate and add py4j zip file inside /opt/spark/python/lib/
py4j_zip = glob.glob('/opt/spark/python/lib/py4j-*.zip')
if py4j_zip:
    sys.path.insert(0, py4j_zip[0])

from pyspark.sql import SparkSession
import boto3

session = boto3.Session(profile_name="data-eng")
creds = session.get_credentials().get_frozen_credentials()

BUCKET_NAME = os.getenv("S3_BUCKET_ARN")

spark = (
    SparkSession.builder.appName("S3JsonSandbox")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.5.0")
    .config("spark.hadoop.fs.s3a.access.key", creds.access_key)
    .config("spark.hadoop.fs.s3a.secret.key", creds.secret_key)
    .config("spark.hadoop.fs.s3a.session.token", creds.token or "")
    .getOrCreate()
)

# Load the raw JSON directly from your S3 data lake bucket
s3_path = f"s3a://{BUCKET_NAME}/raw/transit/vehicles/year=2026/month=07/day=17/hour=02/vehicles.json"
df = spark.read.json(s3_path)

# Inspect the raw nested structure
df.printSchema()
df.show(5, truncate=False)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-abe148ad-79f2-4cf3-a1cf-df515acd06f7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 839ms :: artifacts dl 29ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyticsaccelerator#analyticsaccelerator-

root
 |-- bustime-response: struct (nullable = true)
 |    |-- vehicle: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- des: string (nullable = true)
 |    |    |    |-- dly: boolean (nullable = true)
 |    |    |    |-- hdg: string (nullable = true)
 |    |    |    |-- lat: string (nullable = true)
 |    |    |    |-- lon: string (nullable = true)
 |    |    |    |-- mode: long (nullable = true)
 |    |    |    |-- origtatripno: string (nullable = true)
 |    |    |    |-- pdist: long (nullable = true)
 |    |    |    |-- pid: long (nullable = true)
 |    |    |    |-- psgld: string (nullable = true)
 |    |    |    |-- rt: string (nullable = true)
 |    |    |    |-- stsd: string (nullable = true)
 |    |    |    |-- stst: long (nullable = true)
 |    |    |    |-- tablockid: string (nullable = true)
 |    |    |    |-- tatripid: string (nullable = true)
 |    |    |    |-- tmstmp: string (nullable = true)
 |    |    |    |-- vid: stri

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# 1. Extract the vehicle array out of bustime-response
df_vehicles = df.select(
    explode(col("`bustime-response`.vehicle")).alias("vehicle")
)

# 2. Flatten out the individual vehicle fields
df_flat = df_vehicles.select(
    col("vehicle.vid").alias("vehicle_id"),
    col("vehicle.rt").alias("route"),
    col("vehicle.des").alias("destination"),
    col("vehicle.lat").cast("double").alias("latitude"),
    col("vehicle.lon").cast("double").alias("longitude"),
    col("vehicle.dly").alias("is_delayed"),
    col("vehicle.tmstmp").alias("timestamp"),
)

df_flat.printSchema()
df_flat.show(10, truncate=False)

root
 |-- vehicle_id: string (nullable = true)
 |-- route: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- is_delayed: boolean (nullable = true)
 |-- timestamp: string (nullable = true)

+----------+-----+-----------+------------------+------------------+----------+--------------+
|vehicle_id|route|destination|latitude          |longitude         |is_delayed|timestamp     |
+----------+-----+-----------+------------------+------------------+----------+--------------+
|8937      |20   |Austin     |41.88092041015625 |-87.7115324193781 |false     |20260716 21:46|
|8306      |20   |Austin     |41.8806037902832  |-87.74446868896484|false     |20260716 21:46|
|8300      |20   |Austin     |41.880250460859656|-87.76977284749348|false     |20260716 21:46|
|8603      |20   |Pulaski    |41.88046521273526 |-87.74469188343394|false     |20260716 21:46|
|8569      |20   |Michigan   |41.880475843

In [3]:
# Inspect physical execution plan and wide/narrow dependencies
df_flat.explain(True)

== Parsed Logical Plan ==
'Project ['vehicle.vid AS vehicle_id#16, 'vehicle.rt AS route#17, 'vehicle.des AS destination#18, cast('vehicle.lat as double) AS latitude#19, cast('vehicle.lon as double) AS longitude#20, 'vehicle.dly AS is_delayed#21, 'vehicle.tmstmp AS timestamp#22]
+- Project [vehicle#15]
   +- Generate explode(bustime-response#6.vehicle), false, [vehicle#15]
      +- Relation [bustime-response#6] json

== Analyzed Logical Plan ==
vehicle_id: string, route: string, destination: string, latitude: double, longitude: double, is_delayed: boolean, timestamp: string
Project [vehicle#15.vid AS vehicle_id#16, vehicle#15.rt AS route#17, vehicle#15.des AS destination#18, cast(vehicle#15.lat as double) AS latitude#19, cast(vehicle#15.lon as double) AS longitude#20, vehicle#15.dly AS is_delayed#21, vehicle#15.tmstmp AS timestamp#22]
+- Project [vehicle#15]
   +- Generate explode(bustime-response#6.vehicle), false, [vehicle#15]
      +- Relation [bustime-response#6] json

== Optimized 

In [4]:
df_flat.groupBy("route").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['route], ['route, 'count(1) AS count#53]
+- Project [vehicle#15.vid AS vehicle_id#16, vehicle#15.rt AS route#17, vehicle#15.des AS destination#18, cast(vehicle#15.lat as double) AS latitude#19, cast(vehicle#15.lon as double) AS longitude#20, vehicle#15.dly AS is_delayed#21, vehicle#15.tmstmp AS timestamp#22]
   +- Project [vehicle#15]
      +- Generate explode(bustime-response#6.vehicle), false, [vehicle#15]
         +- Relation [bustime-response#6] json

== Analyzed Logical Plan ==
route: string, count: bigint
Aggregate [route#17], [route#17, count(1) AS count#53L]
+- Project [vehicle#15.vid AS vehicle_id#16, vehicle#15.rt AS route#17, vehicle#15.des AS destination#18, cast(vehicle#15.lat as double) AS latitude#19, cast(vehicle#15.lon as double) AS longitude#20, vehicle#15.dly AS is_delayed#21, vehicle#15.tmstmp AS timestamp#22]
   +- Project [vehicle#15]
      +- Generate explode(bustime-response#6.vehicle), false, [vehicle#15]
         +- Relati

In [5]:
df_flat = df_flat.withColumn('vehicle_id', col('vehicle_id').cast(IntegerType()))
df_flat = df_flat.withColumn('timestamp', to_timestamp(col('timestamp'), 'yyyyMMdd HH:mm'))

In [6]:
df_flat.show(10, truncate=False)

+----------+-----+-----------+------------------+------------------+----------+-------------------+
|vehicle_id|route|destination|latitude          |longitude         |is_delayed|timestamp          |
+----------+-----+-----------+------------------+------------------+----------+-------------------+
|8937      |20   |Austin     |41.88092041015625 |-87.7115324193781 |false     |2026-07-16 21:46:00|
|8306      |20   |Austin     |41.8806037902832  |-87.74446868896484|false     |2026-07-16 21:46:00|
|8300      |20   |Austin     |41.880250460859656|-87.76977284749348|false     |2026-07-16 21:46:00|
|8603      |20   |Pulaski    |41.88046521273526 |-87.74469188343394|false     |2026-07-16 21:46:00|
|8569      |20   |Michigan   |41.880475843274915|-87.77407156454551|false     |2026-07-16 21:45:00|
|8172      |20   |Michigan   |41.8804482199929  |-87.74537714177912|false     |2026-07-16 21:46:00|
|8719      |20   |Michigan   |41.88090377018369 |-87.70926561026738|false     |2026-07-16 21:46:00|
